## Imports

In [47]:
from pydantic import BaseModel, Field
from typing import Optional, List
from ollama import chat
from unidecode import unidecode
import pandas as pd
import tiktoken
import argparse
import json
import os
import re
import hashlib
import csv
import string
import unicodedata

## Get Human Annotated Ground Truth, CTMAP, and Manuscript

In [48]:
ds_name = "adipose_Emont2022"
folder_name = f"../../data/{ds_name}"
evidence_human =f"{folder_name}/evidence_human/evidence.json"
ctmap = f"{folder_name}/ctmap/ctmap.json"
rev_ctmap = f"{folder_name}/ctmap/rev_ctmap.json"
#manuscript = "biorxv.txt"
manuscript = f"{folder_name}/manuscript/manuscript.txt"

with open(rev_ctmap, 'r') as file:
    rev = json.load(file)

In [49]:
def load_evidence(fn, ds_name):
    df = pd.read_json(fn)
    # Unwrap the 'source' column (contains a dictionary) into separate columns
    source_df = df['source'].apply(pd.Series)
    extracted_df = df['extracted'].apply(pd.Series).add_prefix('extracted_')
    # derived_df = df['derived'].apply(pd.Series).add_prefix('derived_')
    df = pd.concat([df.drop(columns=['source', 'extracted']), source_df, extracted_df], axis=1)
    df["ds"] = ds_name
    del df["derived"]
    return df

def short_hash(s: str) -> str:
    """Generate a 7-character hash from a given string."""
    return hashlib.sha256(s.encode()).hexdigest()[:7]

## Run Chunking Code

#### adapted from llm/chunk_filter_merge.ipynb

In [50]:
def split_text_into_windows(corpus, window_size=500, overlap=250, document_name="doc"):
    results = []
    step_size = window_size - overlap

    for start_idx in range(0, len(corpus), step_size):
        end_idx = min(start_idx + window_size, len(corpus))

        extracted_text = corpus[start_idx:end_idx]


        results.append({
            "extracted_text": extracted_text,
            "start_idx": start_idx,
            "end_idx": end_idx,
            "index_type": "character",
            "document_name": document_name
        })

        if end_idx == len(corpus):
            break

    return results

def tokenize_with_offsets(text, encoding="cl100k_base"):
    """Tokenizes text and returns tokens with their character start positions."""
    enc = tiktoken.get_encoding(encoding)
    tokens = []

    etks = enc.encode(text)
    dtks, ptks = enc.decode_with_offsets(etks)

    assert len(etks) == len(ptks)

    for t, p in zip(etks, ptks):
        token = enc.decode_single_token_bytes(t).decode("utf-8")
        tobj = {
            "token": token,
            "enc_token": t,
            "start_idx": p,
            "end_idx": p + len(token),
        }
        tokens.append(tobj)
    return tokens

def split_text_into_windows(corpus, window_size=200, overlap=50, document_name="doc", encoding="cl100k_base"):
    tokens = tokenize_with_offsets(corpus, encoding=encoding)
    results = []

    step_size = window_size - overlap
    for start_idx in range(0, len(tokens), step_size):
        end_idx = min(start_idx + window_size, len(tokens))

        window_tokens = tokens[start_idx:end_idx]

        extracted_text = corpus[window_tokens[0]['start_idx']:window_tokens[-1]['end_idx']]

        # move the LLM checking verification here and if LLM or containsCTL then we can append it (doing it in parallel not series)

        # additional chunking verification

        # containsCTL = False

        # for ky in rev.keys():
        #     if ky in extracted_text.upper().split():
        #         containsCTL = True

        # if (containsCTL):
        results.append({
            "extracted_text": extracted_text,
            "start_idx": window_tokens[0]['start_idx'],
            "end_idx": window_tokens[-1]['end_idx'],
            "index_type": "character",
            "document_name": document_name
        })

        if end_idx == len(tokens):
            break

    return results

In [51]:
def norm_text(text):
    """
    Normalize text to ensure UTF-8 compatibility for NLP processing.

    This function:
    1. Normalizes Unicode to the NFC form
    2. Replaces problematic characters with ASCII equivalents
    3. Handles common special characters that cause issues

    Args:
        text (str): Input text to normalize

    Returns:
        str: Normalized text safe for UTF-8 processing
    """
    if not isinstance(text, str):
        try:
            text = str(text)
        except:
            return ""

    # Step 1: Unicode normalization to NFC form (composed form)
    # This combines characters and diacritics when possible
    text = unicodedata.normalize("NFC", text)
    text = unidecode(text)

    # Step 2: Map specific problematic characters to ASCII equivalents
    # char_map = {
    #     "ɛ": "e",  # epsilon
    #     "ɑ": "a",  # alpha
    #     "β": "b",  # beta
    #     "δ": "d",  # delta
    #     "γ": "g",  # gamma
    #     "λ": "l",  # lambda
    #     "μ": "u",  # mu
    #     "π": "pi",  # pi
    #     "θ": "theta",  # theta
    #     "τ": "t",  # tau
    #     "ω": "omega",  # omega
    #     "′": "'",  # prime
    #     "″": '"',  # double prime
    #     "–": "-",  # en dash
    #     "—": "--",  # em dash
    #     """: "'",        # curly single quote
    #     """: "'",  # curly single quote
    #     '"': '"',  # curly double quote
    #     '"': '"',  # curly double quote
    #     "…": "...",  # ellipsis
    #     "•": "*",  # bullet
    #     "·": ".",  # middle dot
    #     "×": "x",  # multiplication sign
    #     "÷": "/",  # division sign
    #     "≤": "<=",  # less than or equal
    #     "≥": ">=",  # greater than or equal
    #     "≠": "!=",  # not equal
    #     "≈": "~",  # approximately equal
    #     "∞": "inf",  # infinity
    #     "∂": "d",  # partial differential
    #     "∫": "integral",  # integral
    #     "∑": "sum",  # sum
    #     "∏": "product",  # product
    #     "√": "sqrt",  # square root
    #     "∝": "prop to",  # proportional to
    #     "∠": "angle",  # angle
    #     "△": "triangle",  # triangle
    #     "□": "square",  # square
    #     "∈": "in",  # element of
    #     "∉": "not in",  # not an element of
    #     "⊂": "subset",  # subset
    #     "⊃": "superset",  # superset
    #     "∪": "union",  # union
    #     "∩": "intersect",  # intersection
    #     "⊆": "subseteq",  # subset or equal
    #     "⊇": "superseteq",  # superset or equal
    #     "Π": "Pi",  # uppercase pi
    #     "⁄": "/",  # fraction slash
    #     "½": "1/2",
    #     "⅓": "1/3",
    #     "⅔": "2/3",
    #     "¾": "3/4",
    #     "⅕": "1/5",
    #     "⅖": "2/5",
    #     "⅗": "3/5",
    #     "⅘": "4/5",
    #     "⅙": "1/6",
    #     "⅚": "5/6",
    #     "⅛": "1/8",
    #     "⅜": "3/8",
    #     "⅝": "5/8",
    #     "⅞": "7/8",
    #     "ὀ": "o",  # omicron with smooth breathing
    #     "ξ": "x",  # xi
    #     "ύ": "y",  # upsilon with tonos
    #     "ς": "s",  # final sigma
    #     "ε": "e",  # epsilon
    #     "ν": "n",  # nu
    #     "ή": "e",  # eta with tonos
    #     "ἄ": "a",  # alpha with smooth breathing and tonos
    #     "ζ": "z",  # zeta
    #     "ο": "o",  # omicron
    #     "ᵻ": "i",  # Latin letter with stroke
    #     "ɒ": "a",  # turned alpha
    #     "ə": "e",  # schwa
    #     "ɔ": "o",  # open-mid back rounded vowel
    #     "ː": ":",  # IPA length mark
    #     "κ": "k",  # kappa
    #     "ί": "i",  # iota with tonos
    #     "φ": "ph",  # phi
    #     "ω": "o",  # omega
    # }

    char_map = {
        "–": "-",  # en dash
        "—": "--",  # em dash
        "‘": "'",  # left single quote
        "’": "'",  # right single quote
        "“": '"',  # left double quote
        "”": '"',  # right double quote
        "…": "...",  # ellipsis
        "•": "*",  # bullet
        "·": ".",  # middle dot
        "×": "x",  # multiplication sign
        "÷": "/",  # division sign
        "≤": "<=",  # less than or equal
        "≥": ">=",  # greater than or equal
        "≠": "!=",  # not equal
        "≈": "~",  # approximately equal
        "∞": "inf",  # infinity
        "∂": "d",  # partial differential
        "∫": "integral",  # integral
        "∑": "sum",  # sum
        "∏": "product",  # product
        "√": "sqrt",  # square root
        "∝": "prop to",  # proportional to
        "∠": "angle",  # angle
        "△": "triangle",  # triangle
        "□": "square",  # square
        "∈": "in",  # element of
        "∉": "not in",  # not an element of
        "⊂": "subset",  # subset
        "⊃": "superset",  # superset
        "∪": "union",  # union
        "∩": "intersect",  # intersection
        "⊆": "subseteq",  # subset or equal
        "⊇": "superseteq",  # superset or equal
    }

    for char, replacement in char_map.items():
        text = text.replace(char, replacement)
    # Step 3: Remove any remaining non-ASCII characters (optional)
    # Uncomment if you want to remove ALL non-ASCII characters
    # text = re.sub(r'[^\x00-\x7F]+', '', text)

    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)  # .strip()

    return text


In [52]:
with open(manuscript, "r") as f:
    paper = f.read().strip()

# Split the text into windows
windows = split_text_into_windows(norm_text(paper), window_size=100, overlap=10, document_name=ds_name)

In [53]:
windows[-1]

{'extracted_text': "version 0.1.2) using the Hamiltonian Monte Carlo sampling method. The model formula used was 'Depot + BMI' (human) or 'Depot + Diet') (mouse) for all objects in for which both of these covariates were present, or the individual covariate when only a single condition was present. Reporting summary Further information on research design is available in the Nature Research Reporting Summary linked to this paper.",
 'start_idx': 65581,
 'end_idx': 65993,
 'index_type': 'character',
 'document_name': 'adipose_Emont2022'}

## Run LLM Against the Chunks (T/F)

In [54]:
# Permissive class object (allows for missing fields)
class GeneFinder(BaseModel):
    hasCelltypeGene: bool = Field(False, description="Does the text discuss biological cell type marker genes?")

In [55]:
def extract_genes(user_content, system_prompt, data_model, model="llama3.2"):
    response = chat(
        messages=[
            {
                'role': 'system',
                'content': system_prompt,
            },
            {
                'role': 'user',
                'content': user_content,
            }
        ],
        model = model,
        format = data_model.model_json_schema(),
        options={'temperature': 0},  # Make responses more deterministic

    )
    genes = data_model.model_validate_json(response.message.content)
    return genes.model_dump()

system_prompt = """
You are an expert biologist specialized in identifying discussions about cell types and their associated marker genes in scientific texts. Your job is to determine if the provided text discusses relationships between cell types and their marker genes. The text may contain scientific jargon, abbreviations, and complex sentences. Your task is to identify whether the text discusses cell types and their marker genes.

Does the following text discuss relationships between cell types and their marker genes?

Criteria for “True”:
- Mentions or implies cell types AND specifically mentions a gene or set of genes.
- Discusses marker genes as identifying, characterizing, or differentiating particular cell types.

Otherwise, return “False”.
"""

In [56]:
check = []
for idx, window in enumerate(windows):
    genes = extract_genes(window["extracted_text"], system_prompt, GeneFinder, model="llama3.2")
    if idx % 50 == 0:
        print(f"Processed {idx} rows.")
    window.update({"hasCelltypeGene": genes["hasCelltypeGene"]})
    check.append(window)

Processed 0 rows.
Processed 50 rows.
Processed 100 rows.
Processed 150 rows.


## Filter out False Chunks

In [57]:
filt = [i for i in check if i["hasCelltypeGene"]]
len(filt)

88

## Merge Adjacent Chunks

In [58]:
merged = []

current = filt[0]

for next_row in filt[1:]:
    if current['end_idx'] > next_row['start_idx']:
        # Calculate the non-overlapping part
        overlap_len = current['end_idx'] - next_row['start_idx']
        current['extracted_text'] += next_row['extracted_text'][overlap_len:]
        current['end_idx'] = next_row['end_idx']
    else:
        merged.append(current)
        current = next_row.copy()

merged.append(current)

In [59]:
merged[2]

{'extracted_text': ' of all 197,721 sequenced mouse cells split by diet. g, Marker genes for each cell population in the mouse WAT dataset. Full size image[] Fig. 1: A single-cell atlas of human white adipose tissue. An atlas of mouse white adipose tissue Mouse models are commonly used to study adipose tissue biology14. We thus sought to compare mouse and human WAT at the single-cell level by performing sNuc-seq on inguinal adipose tissue (',
 'start_idx': 3161,
 'end_idx': 3585,
 'index_type': 'character',
 'document_name': 'adipose_Emont2022',
 'hasCelltypeGene': True}

In [60]:
def reformat_merged(merged):
    reformatted = []
    for m in merged:
        reformatted.append({
            "extracted": {
                "organism": "",
                "cell_type_label": "",
                "cell_source": "",
                "cell_state": "",
                "feature_name": "",
                "feature_type": "",
            },
            "derived": {
                "organism": "",
                "cell_type_id": "",
                "cell_type_label": "",
                "cell_source": "",
                "cell_state": "",
                "feature_name": "",
                "feature_type": "",
                "feature_identifier": "",
                "feature_identifier_type": "",
            },
            "source": {
                "source_rationale": m["extracted_text"],
                "source_type": "text",
                "source_id": "text"
            }
        })
    return reformatted

In [61]:
final_set = reformat_merged(merged)

In [62]:
uniq_src = set([obj['source']['source_rationale'] for obj in final_set])

## Extract CTMGs with LLM

In [63]:
# Permissive class object (allows for missing fields)
class MarkerGene(BaseModel):
    marker_gene_name: Optional[str] = Field(
        None, 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: Optional[str] = Field(
        None, 
        description="The name of the cell type for which this gene serves as a marker."
    )

class MarkerGeneList(BaseModel):
    genes: List[MarkerGene]  # A list of marker genes extracted from the input text

# Restrictive class object does not allow for missing fields
class MarkerGeneStrict(BaseModel):
    marker_gene_name: str = Field(
        ..., 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: str = Field(
        ..., 
        description="The name of the cell type for which this gene serves as a marker."
    )

class MarkerGeneListStrict(BaseModel):
    genes: List[MarkerGeneStrict]  # A list of marker genes extracted from the input text

In [64]:
MODEL_TO_USE = "llama3.2"

LLMODELS = {
    "deepseek-r1": "deepseek-r1",
    "llama3.2": "llama3.2",
    "llama3.3": "llama3.3"
}

DATA_MODELS = {
    "MarkerGeneListStrict": MarkerGeneListStrict,
    "MarkerGeneStrict": MarkerGeneStrict
}

system_prompt = """
You are an expert in genomics analyzing scientific literature to extract marker genes for different cell types. 

Your goal is to identify and structure marker gene data from the given text. For each marker gene mentioned, extract:
- The **gene name** (marker_gene_name).
- The **cell type** it is associated with (cell_type_name).

The data must be extracted as written in the text, without any modifications.

Return the results in **structured JSON format** with the following schema:
{
    "genes": [
        {
            "cell_type_name": "Neuron",
            "marker_gene_name": "GeneX",
        },
        ...
    ]
}
"""

model_metadata = {
        "model_id" : LLMODELS[MODEL_TO_USE],
        "system_prompt": system_prompt,
        "system_prompt_hash": short_hash(system_prompt),
        "data_model": DATA_MODELS["MarkerGeneListStrict"].__name__
}

In [65]:
def is_valid(ls, source) :
    for obj in ls:
        if obj["cell_type_name"].upper() not in source.upper() or obj["marker_gene_name"].upper() not in source.upper():
            return {obj["cell_type_name"].upper(): obj["cell_type_name"].upper() not in source.upper(), obj["marker_gene_name"].upper():  obj["marker_gene_name"].upper() not in source.upper()}
    return {"cell_type": True, "marker_gene": True}

In [70]:
def extract_genes(user_prompt, system_prompt, data_model, model=MODEL_TO_USE):
    return_val = list()
    error_msg = ""

    while len(return_val) == 0 or not all(list(is_valid(return_val, user_prompt).values())):
        incorrect_ctmgs = is_valid(return_val, user_prompt)
        
        if all(list(incorrect_ctmgs.values())):
            break
        
        # first, edit error_msg
        ctmgs = list(incorrect_ctmgs.keys())
        if incorrect_ctmgs[ctmgs[0]]:
            error_msg += f" You extracted the cell type name incorrectly. {ctmgs[0]} is the cell type name you extracted, but it is not extracted exactly as it appears within the user prompt. Redo to ensure you are extracting the cell type name EXACTLY as it appears in the given prompt."
        if incorrect_ctmgs[ctmgs[1]]:
            error_msg += f" You extracted the marker gene name incorrectly. {ctmgs[1]} is the marker gene name you extracted, but it is not extracted exactly as it appears within the user prompt. Redo to ensure you are extracting the marker gene name EXACTLY as it appears in the given prompt."
        
        # user_prompt is the source rationale
        response = chat(
            messages=[
                {
                    'role': 'system',
                    'content': system_prompt,
                },
                {
                    'role': 'user',
                    'content': user_prompt + error_msg,
                }
            ],
            model = model,
            format = data_model.model_json_schema(),
            options={'temperature': 0},  # Make responses more deterministic

        )

        genes = data_model.model_validate_json(response.message.content)
        return_val = genes.model_dump()["genes"]
        print(return_val)

    return return_val

In [71]:
ct_mgs = []

for idx, text in enumerate(uniq_src):
    print(text)
    print(f"{idx + 1}/{len(uniq_src)}")
    ct_mg = extract_genes(text, model_metadata["system_prompt"], DATA_MODELS[model_metadata["data_model"]], model=model_metadata["model_id"])
    
    if len(ct_mg) == 0:
        continue
        
    for g in ct_mg:
        tmp = {
            "extracted": {
                "organism": "homo_sapiens",
                "cell_type_label": g["cell_type_name"],
                "cell_source": None,
                 "cell_state": None,
                "feature_name": g["marker_gene_name"],
                "feature_type": "gene",
            },
            "derived": {
                "organism": "homo_sapiens",
                "cell_type_id": None,
                "cell_type_label": g["cell_type_name"],
                "cell_source": None,
                "cell_state": None,
                "feature_name": g["marker_gene_name"],
                "feature_type": "gene",
                "feature_identifier": None,
                "feature_identifier_type": None,
            },
            "source": {
                "source_type": "text",
                "source_rationale": text,
                "source_id": "text"
            }
        }

        ct_mgs.append(tmp)

/unspliced RNA ratio. The remaining clusters were annotated based on marker gene expression. In some cases, smaller subclusters (T and NK cells, B cells, monocytes/neutrophils) were further subset and PCA and clustering analysis but not integration was re-run in order to assign clusters. After subcluster annotation, identities were mapped back onto the original object and cells that were removed from the subclustered objects were similarly removed from the all-cell object. Deconvolution of bulk RNA-se
1/20
 of all 197,721 sequenced mouse cells split by diet. g, Marker genes for each cell population in the mouse WAT dataset. Full size image[] Fig. 1: A single-cell atlas of human white adipose tissue. An atlas of mouse white adipose tissue Mouse models are commonly used to study adipose tissue biology14. We thus sought to compare mouse and human WAT at the single-cell level by performing sNuc-seq on inguinal adipose tissue (
2/20
 using a Louvain algorithm, 40 principal components, and a

In [72]:
ct_mgs

[]

In [69]:
llm_folder = os.path.join(folder_name, f"workflow_evidence_llm_{model_metadata['model_id']}_{model_metadata['data_model']}_{model_metadata['system_prompt_hash']}")
os.mkdir(llm_folder)

output_path = os.path.join(llm_folder, "evidence.json") 
with open(output_path, 'w') as f:
    json.dump(ct_mgs, f, indent = 4)

model_fn = os.path.join(llm_folder, "model_metadata.json")
with open(model_fn, 'w') as f:
    json.dump(model_metadata, f, indent = 4)

FileExistsError: [Errno 17] File exists: '../../data/adipose_Emont2022/workflow_evidence_llm_llama3.2_MarkerGeneListStrict_4efcc22'

## Map Feature Names to Feature Identifiers

In [53]:
def update_json_with_identifiers(json_path, tsv_path, output_path):
    # Step 1: Load the feature mapping, uppercasing keys
    feature_map = {}
    with open(tsv_path, 'r') as tsvfile:
        reader = csv.reader(tsvfile, delimiter='\t')
        for row in reader:
            if len(row) >= 2:
                key = row[0].strip().upper()
                value = row[1].strip()
                feature_map[key] = value

    # Step 2: Load JSON
    with open(json_path, 'r') as f:
        data = json.load(f)

    # Step 3: Process and conditionally update
    for entry in data:
        derived = entry.get("derived", {})
        
        organism = derived.get("organism", "").strip().lower()
        if derived.get("feature_name", "") is str:
            feature_name = derived.get("feature_name", "").strip().upper()
        else:
            feature_name = derived.get("feature_name", "")
        if organism == "homo_sapiens" and feature_name in feature_map:
            derived["feature_identifier"] = feature_map[feature_name]
            derived["feature_identifier_type"] = "ensembl"

    # Step 4: Save updated JSON
    with open(output_path, 'w') as f:
        json.dump(data, f, indent=2)

In [54]:
gmap = "../../data/gmap.txt"

update_json_with_identifiers(output_path, gmap, output_path)

## Map Cell Type Labels to Cell Type Identifiers

In [55]:
def find_key_given_value(label_map: dict, label):
    for key in label_map.keys():
        if label in label_map[key]:
            return key
    return None

def update_json(label_map, json_fn = 'evidence.json'):
    with open(json_fn, 'r') as file:
        data = json.load(file)
        for obj in data:
            if obj['derived']['cell_type_label'] is not None:
                obj['derived']['cell_type_id'] = find_key_given_value(label_map, obj['derived']['cell_type_label'].upper().strip())
        with open(json_fn, "w") as file:
            json.dump(data, file, indent = 4)
    
    print("Processing complete!")

In [56]:
with open(ctmap, 'r') as f:
    ctmap_json = json.load(f)

update_json(ctmap_json, output_path)

Processing complete!
